# 49 - Open Simulators And Synthetic Data

## What / Why / How

**What we are trying to do:** Compare open simulators and synthetic-data engines: Isaac Lab, MuJoCo, ManiSkill, RoboCasa365, RoboTwin, Genesis, LIBERO, and CALVIN.

**Why this matters:** Simulation is where most robot learning scales, but every simulator has a bias. Choosing the wrong one can waste months.

**How we will do it:** Score simulators by task fit, speed, photorealism, contact quality, data generation, and sim-to-real risk.

## Simulator Selection

Simulation is not one thing. Locomotion, dexterous contact, photorealistic perception, mobile manipulation, and benchmark reproducibility all favor different tools.

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
simulators = [
    {"name": "Isaac Lab", "speed": 0.9, "photo": 0.8, "contact": 0.75, "robot_learning": 0.95, "ease": 0.45},
    {"name": "MuJoCo", "speed": 0.85, "photo": 0.35, "contact": 0.85, "robot_learning": 0.8, "ease": 0.75},
    {"name": "ManiSkill", "speed": 0.9, "photo": 0.65, "contact": 0.8, "robot_learning": 0.9, "ease": 0.7},
    {"name": "RoboCasa365", "speed": 0.65, "photo": 0.8, "contact": 0.65, "robot_learning": 0.85, "ease": 0.55},
    {"name": "RoboTwin 2.0", "speed": 0.75, "photo": 0.65, "contact": 0.75, "robot_learning": 0.8, "ease": 0.6},
    {"name": "Genesis", "speed": 0.95, "photo": 0.65, "contact": 0.7, "robot_learning": 0.8, "ease": 0.75},
    {"name": "Gazebo Sim", "speed": 0.45, "photo": 0.55, "contact": 0.55, "robot_learning": 0.45, "ease": 0.65},
]

task_weights = {"speed": 0.25, "photo": 0.15, "contact": 0.25, "robot_learning": 0.25, "ease": 0.10}
for sim in sorted(simulators, key=lambda s: sum(s[k]*w for k, w in task_weights.items()), reverse=True):
    score = sum(sim[k] * w for k, w in task_weights.items())
    print(f"{score:.3f} {sim['name']}")

In [ ]:
# Domain randomization coverage toy model.
rng = np.random.default_rng(49)
train_friction = rng.uniform(0.4, 1.2, 1000)
real_friction = rng.normal(0.75, 0.18, 500)
real_friction = np.clip(real_friction, 0.1, 1.5)
covered = np.mean((real_friction >= train_friction.min()) & (real_friction <= train_friction.max()))
print("real friction covered by train range:", covered)

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(7, 3))
    plt.hist(train_friction, bins=30, alpha=0.5, label="train randomized")
    plt.hist(real_friction, bins=30, alpha=0.5, label="real estimate")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Change weights for a vision-heavy task.
2. Add Webots or Drake to the simulator table.
3. Write a simulator choice memo for humanoid locomotion versus kitchen manipulation.